<a href="https://colab.research.google.com/github/MohamedAamer01/otto-recommender-sys/blob/main/notebooks/03_recsys_theory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Théorie des systèmes de recommandation

## Bloc 1 : popularité, filtrage collaboratif user-based et item-based

### Le cadre : la matrice d'interactions R
Un système de recommandation travaille sur une matrice **R** (lignes = "utilisateurs", colonnes = articles).
R[u, i] = 1 si l'utilisateur u a interagi avec l'article i, 0 sinon.
Objectif : pour un utilisateur u, ordonner les articles non vus et garder les 20 meilleurs.

**Dans OTTO** : une ligne = une *session* (pas d'identité persistante).

In [1]:
n_sessions = 12_899_779
n_items    = 1_855_603
mean_distinct = 10.36327                      # mesuré au notebook 2

n_pairs = mean_distinct * n_sessions          # nombre de cases non nulles dans R
n_cells = n_sessions * n_items

print(f"Cases non nulles : {n_pairs:,.0f}")
print(f"Cases totales    : {n_cells:.3e}")
print(f"Densité          : {n_pairs / n_cells * 100:.5f} %")
print(f"R dense en 1 octet/case : {n_cells / 1e12:.1f} To")

Cases non nulles : 133,683,893
Cases totales    : 2.394e+13
Densité          : 0.00056 %
R dense en 1 octet/case : 23.9 To


**les trois méthodes sur une mini-matrice**

In [2]:
import numpy as np
import pandas as pd

items = ["A", "B", "C", "D", "E", "F"]
R = pd.DataFrame(
    [[1,1,0,0,0,0],   # s0
     [1,1,1,0,0,0],   # s1
     [0,1,1,1,0,0],   # s2
     [0,0,0,1,1,1],   # s3
     [1,0,1,0,0,0]],  # s4
    columns=items, index=[f"s{i}" for i in range(5)])
Rm = R.values.astype(float)
display(R)

def recommend(scores, seen, k=3):
    s = pd.Series(scores, index=items).drop(seen)      # on retire les articles déjà vus
    return s.sort_values(ascending=False).head(k)

def run(query_items):
    q = np.array([1.0 if it in query_items else 0.0 for it in items])

    # 1) popularité
    pop = Rm.sum(axis=0)

    # 2) user-based : cosinus entre la requête et chaque session
    sim_users = (Rm @ q) / (np.linalg.norm(Rm, axis=1) * np.linalg.norm(q))
    user_scores = sim_users @ Rm

    # 3) item-based : cosinus entre colonnes
    col_norm = np.linalg.norm(Rm, axis=0)
    S = (Rm.T @ Rm) / np.outer(col_norm, col_norm)
    np.fill_diagonal(S, 0)                              # un article n'est pas son propre voisin
    item_scores = q @ S

    print(f"\n=== Session qui a vu {query_items} ===")
    print("Popularité :\n", recommend(pop, query_items).round(3))
    print("User-based :\n", recommend(user_scores, query_items).round(3))
    print("Item-based :\n", recommend(item_scores, query_items).round(3))

run(["A"])
run(["E"])

,A,B,C,D,E,F
s0,1,1,0,0,0,0
s1,1,1,1,0,0,0
s2,0,1,1,1,0,0
s3,0,0,0,1,1,1
s4,1,0,1,0,0,0



=== Session qui a vu ['A'] ===
Popularité :
 B    3.0
C    3.0
D    2.0
dtype: float64
User-based :
 B    1.284
C    1.284
D    0.000
dtype: float64
Item-based :
 B    0.667
C    0.667
D    0.000
dtype: float64

=== Session qui a vu ['E'] ===
Popularité :
 A    3.0
B    3.0
C    3.0
dtype: float64
User-based :
 F    0.577
D    0.577
A    0.000
dtype: float64
Item-based :
 F    1.000
D    0.707
A    0.000
dtype: float64
